## Portfolio 1

Wij hebben als spel 4 op een rij gekozen.

In [2]:
import sys
!"{sys.executable}" -m pip install pettingzoo chess pygame numpy matplotlib ipykernel


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip


In [3]:
from pettingzoo.classic import connect_four_v3

In [4]:
env = connect_four_v3.env(render_mode="human")
env.reset()

for agent in env.agent_iter():

    observation, reward, termination, truncation, info = env.last()

    if termination or truncation:
        action = None
    else:
        mask = observation["action_mask"]

        # kies eerste geldige zet
        legal_moves = [i for i, m in enumerate(mask) if m == 1]
        action = legal_moves[0]

    env.step(action)
    

In [5]:
import numpy as np
import random

class BlockingAgent:

    def act(self, observation):

        board = observation["observation"]
        mask = observation["action_mask"]

        legal_moves = [i for i, m in enumerate(mask) if m == 1]

        opponent = board[:, :, 1]

        # 🔥 check elke kolom: voorkomt directe winst
        for col in legal_moves:

            row = self.get_next_open_row(opponent, col)
            if row is None:
                continue

            temp_board = opponent.copy()
            temp_board[row][col] = 1

            if self.check_win(temp_board):
                print("BLOCKING WIN:", col)
                return col

        # 🔥 extra: detecteer 3-op-een-rij dreigingen
        for col in legal_moves:

            row = self.get_next_open_row(opponent, col)
            if row is None:
                continue

            if self.creates_threat(opponent, row, col):
                print("BLOCKING THREAT:", col)
                return col

        return random.choice(legal_moves)

    def get_next_open_row(self, board, col):
        for row in reversed(range(6)):
            if board[row][col] == 0:
                return row
        return None

    def check_win(self, board):

        # horizontaal
        for r in range(6):
            for c in range(4):
                if all(board[r][c+i] == 1 for i in range(4)):
                    return True

        # verticaal
        for r in range(3):
            for c in range(7):
                if all(board[r+i][c] == 1 for i in range(4)):
                    return True

        # diagonaal \
        for r in range(3):
            for c in range(4):
                if all(board[r+i][c+i] == 1 for i in range(4)):
                    return True

        # diagonaal /
        for r in range(3, 6):
            for c in range(4):
                if all(board[r-i][c+i] == 1 for i in range(4)):
                    return True

        return False

    def creates_threat(self, board, row, col):

        temp = board.copy()
        temp[row][col] = 1

        # tel aantal 3-op-een-rij
        count = 0

        # horizontaal
        for c in range(max(0, col-3), min(4, col+1)):
            window = temp[row][c:c+4]
            if np.sum(window) == 3:
                count += 1

        # verticaal
        for r in range(max(0, row-3), min(3, row+1)):
            window = [temp[r+i][col] for i in range(4)]
            if sum(window) == 3:
                count += 1

        # diagonaal \
        for i in range(-3, 1):
            coords = [(row+i+j, col+i+j) for j in range(4)]
            if all(0 <= r < 6 and 0 <= c < 7 for r, c in coords):
                window = [temp[r][c] for r, c in coords]
                if sum(window) == 3:
                    count += 1

        # diagonaal /
        for i in range(-3, 1):
            coords = [(row-i-j, col+i+j) for j in range(4)]
            if all(0 <= r < 6 and 0 <= c < 7 for r, c in coords):
                window = [temp[r][c] for r, c in coords]
                if sum(window) == 3:
                    count += 1

        return count > 0

In [ ]:
from pettingzoo.classic import connect_four_v3
#from agents.blocking_agent import BlockingAgent

env = connect_four_v3.env(render_mode="human")
env.reset()

agent_ai = BlockingAgent()

for agent in env.agent_iter():

    observation, reward, termination, truncation, info = env.last()

    if termination or truncation:
        action = None
    else:
        mask = observation["action_mask"]
        legal_moves = [i for i, m in enumerate(mask) if m == 1]

        if agent == "player_0":
            # 👤 JIJ speelt
            print("\nJouw beurt!")
            print("Kies kolom (0-6): ", legal_moves)

            while True:
                try:
                    action = int(input("Jouw zet: "))
                    if action in legal_moves:
                        break
                    else:
                        print("❌ Ongeldige zet, kies uit:", legal_moves)
                except:
                    print("❌ Voer een getal in")

        else:
            # 🤖 Blocking Agent
            action = agent_ai.act(observation)
            print(f"AI speelt kolom: {action}")

    env.step(action)

env.close()


Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
❌ Voer een getal in
AI speelt kolom: 1

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
BLOCKING THREAT: 0
AI speelt kolom: 0

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]
BLOCKING WIN: 0
AI speelt kolom: 0

Jouw beurt!
Kies kolom (0-6):  [0, 1, 2, 3, 4, 5, 6]


: 